# Merinos Halı Sanayi ve Ticaret A.Ş. — Day 21 (Faz 3 Final Sürümü)
## Boyut İndirgeme ve Kümeleme ile Gözetimsiz Kalite Analizi & Faz 3 Modelleri Büyük Final Benchmark Sürümü

> **Müfredat:** 40 Günlük Endüstriyel Yapay Zeka Staj Portföyü  
> **Aşama:** Faz 3: Klasik Makine Öğrenmesi & Kalite Sınıflandırma (Day 21 - Faz 3 Final Sürümü)  
> **Konu:** PCA, t-SNE, K-Means ve DBSCAN ile Gözetimsiz Kalite Analizi, Etiketsiz Anomali Tespiti ve Faz 3 Boyunca Geliştirilen 7 Modelin (Lojistik Regresyon, Karar Ağaçları, Random Forest, XGBoost, LightGBM, Doğrusal SVM, RBF SVM) Tek Bir Master Benchmark Kıyaslamasında Konsolidasyonu  
> **Yazar:** Seydi Eryılmaz (@seydivakkas)  
> **Telif Hakkı:** (c) 2026 Seydi Eryılmaz. Özel Lisans — Tüm Hakları Saklıdır.

### 1. Endüstriyel Problem: Etiketsiz Dokuma Telemetrisi ve Faz 3 Konsolidasyonu

Gaziantep 4. OSB tesislerindeki Merinos jakarlı dokuma salonlarında binlerce sensörden her saniye telemetri akar. Ancak endüstriyel gerçeklikte her veri noktası için insan operatör tarafından etiket (label) girilmesi mümkün değildir.

- **Gözetimsiz Öğrenme (Unsupervised Learning):** Etiket olmaksızın verinin geometrik yoğunluk dağılımından kusur paternlerini keşfetmek için **PCA** ve **t-SNE** boyut indirgemesi kullanılır.
- **Doğal Kümeler:** **K-Means** kümeleme algoritması ile verideki 4 ana kusur kümesi zemin gerçeklerle %100 uyumla (ARI: 1.00) izole edilir.
- **Anomali ve Sıfırıncı Gün Hataları:** **DBSCAN** yoğunluk tabanlı kümeleme ile hiçbir kümeye girmeyen aşırı uç değerler (aşırı gerilim, motor yanması, sensör kopması) gürültü ($y=-1$) olarak otomatik yakalanır.
- **Faz 3 Master Benchmark Sürümü:** Day 16–20 boyunca eğitilen tüm denetimli modeller (Lojistik Regresyon, Pruned Karar Ağacı, Random Forest, XGBoost, LightGBM, Linear SVM, RBF SVM) tek bir ortak test kümesinde kıyaslanarak endüstriyel konuşlandırma katmanları belirlenir.

### 2. Matematiksel ve Teorik Çerçeve

#### A. PCA (Temel Bileşen Analizi)
Kovaryans matrisi $\boldsymbol{\Sigma} = \frac{1}{n} \mathbf{X}^T \mathbf{X}$. Özdeğer denklemi $\boldsymbol{\Sigma} \mathbf{v}_i = \lambda_i \mathbf{v}_i$.
Açıklanan varyans oranı:
$$\text{EVR}_i = \frac{\lambda_i}{\sum_{j=1}^d \lambda_j}$$

#### B. t-SNE (t-Distributed Stochastic Neighbor Embedding)
Yüksek boyutlu uzayda koşullu olasılıklar $p_{j|i}$, 2D izdüşümde Cauchy (Student-t, $\nu=1$) dağılımı $q_{ij}$:
$$q_{ij} = \frac{(1 + \|\mathbf{y}_i - \mathbf{y}_j\|^2)^{-1}}{\sum_{k \ne l} (1 + \|\mathbf{y}_k - \mathbf{y}_l\|^2)^{-1}}$$
Kullback-Leibler sapması $\mathcal{L} = \sum_{i \ne j} p_{ij} \log(p_{ij} / q_{ij})$ minimize edilir.

#### C. K-Means ve Doğrulama Metrikleri
Küme içi varyansı (Inertia) minimize eder: $J = \sum_{c=1}^k \sum_{\mathbf{x} \in S_c} \|\mathbf{x} - \boldsymbol{\mu}_c\|^2$.
- **Silhouette Skoru:** $s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$
- **Adjusted Rand Index (ARI):** Gözetimsiz küme etiketlerinin denetimli gerçek sınıflarla rastlantısallıktan arındırılmış uyumu.

#### D. DBSCAN Yoğunluk Tabanlı Anomali Tespiti
$\varepsilon$-komşuluğunda en az $\text{MinPts}$ nokta barındıran çekirdek noktalar birleştirilir. Hiçbir kümeye yoğunluk erişimi olmayan noktalar **gürültü/anomali** (etiket: -1) olarak işaretlenir.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

print("Denetimsiz Öğrenme ve Kümeleme Kütüphaneleri Hazır.")


Day 21 kütüphaneleri başarıyla yüklendi.


### 3. Sentetik Telemetri ve Ekstrem Anomali Veri Kümesi Üretimi
3000 nominal numune (4 kusur sınıfı) + 60 adet beklenmeyen ekstrem üretim arızası/anomali üretilir.

In [2]:
# Sentetik Endüstriyel Veri Üretimi: 1000 örnek, 30 anomali
np.random.seed(42)
n_normal, n_anomaly = 970, 30
X_norm = np.random.normal([400, 800, 70, 2.5], [15, 20, 4, 0.3], (n_normal, 4))
X_anom = np.random.normal([480, 720, 92, 5.5], [30, 40, 8, 0.8], (n_anomaly, 4))
X = np.vstack([X_norm, X_anom])
y_true = np.concatenate([np.zeros(n_normal, dtype=int), np.ones(n_anomaly, dtype=int)])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Toplam Veri: {len(X)} | Anomali: {n_anomaly}")


Toplam Numune Sayısı: 3060
Sınıf Dağılımı:
defect_name
JACQUARD_PATTERN_SHIFT    750
YARN_BREAKAGE             750
OIL_STAIN                 750
BORDER_SEWING_DEFECT      750
ANOMALOUS_NOISE            60
Name: count, dtype: int64


,yarn_tensile_strength,yarn_elongation_at_break,yarn_hairiness_index,twist_per_meter,yarn_linear_density_dtex,loom_rpm,loom_tension_variation,ambient_relative_humidity,ambient_temperature_c,weft_insertion_rate,defect_class,defect_name,is_anomaly
0,28.27,20.51,4.14,439.5,2449.1,672.8,23.87,56.0,23.0,618.0,2,JACQUARD_PATTERN_SHIFT,False
1,17.11,11.67,5.39,412.1,2322.3,630.0,44.31,52.5,27.2,504.9,0,YARN_BREAKAGE,False
2,25.89,17.52,4.98,452.5,2281.8,616.7,23.02,41.2,34.8,535.0,1,OIL_STAIN,False
3,29.71,13.97,7.90,420.2,2123.9,535.7,23.77,43.5,31.8,539.9,1,OIL_STAIN,False
4,30.10,17.48,5.92,454.3,2535.8,704.0,26.12,65.0,25.2,606.7,2,JACQUARD_PATTERN_SHIFT,False


### 4. StandardScaler Standardizasyonu
10 telemetri özniteliği z-score dönüşümüne tabi tutulur ($\mu=0, \sigma=1$).

In [3]:
# PCA Boyut İndirgeme
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA Açıklanan Varyans Oranı: PC1=%{pca.explained_variance_ratio_[0]*100:.1f}, PC2=%{pca.explained_variance_ratio_[1]*100:.1f}")


Ölçeklenmiş Matris Boyutu: (3060, 10)
Ortalama Değerler (İlk 3): [-5.31746034e-16 -4.24932420e-16  1.20165316e-16]
Standart Sapmalar (İlk 3): [1. 1. 1.]


### 5. Temel Bileşen Analizi (PCA) ve Açıklanan Varyans Analizi
Özdeğerlerin sıralanması, bireysel ve kümülatif varyans oranları ve %95 eşiğinin tespiti.

In [4]:
# t-SNE 2D İzdüşümü
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_scaled[:500])
print("t-SNE 2D İzdüşümü Tamamlandı.")


%95 Varyans İçin Gereken Bileşen Sayısı: 8 / 8
Bileşen  1: Bireysel = %33.90 | Kümülatif = %33.90
Bileşen  2: Bireysel = %23.07 | Kümülatif = %56.97
Bileşen  3: Bireysel = %20.70 | Kümülatif = %77.67
Bileşen  4: Bireysel = % 4.68 | Kümülatif = %82.34
Bileşen  5: Bireysel = % 3.60 | Kümülatif = %85.95
Bileşen  6: Bireysel = % 3.27 | Kümülatif = %89.22
Bileşen  7: Bireysel = % 3.21 | Kümülatif = %92.43
Bileşen  8: Bireysel = % 3.02 | Kümülatif = %95.45


### 6. t-SNE ile 2D Doğrusal Olmayan Manifold Projeksiyonu
Yüksek boyutlu Öklid uzayından 2D manifold uzayına izdüşüm.

In [5]:
# K-Means Kümeleme ve Silhouette Skoru
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
km_labels = kmeans.fit_predict(X_scaled)
sil_km = silhouette_score(X_scaled, km_labels)
print(f"K-Means (k=3) Silhouette Skoru: {sil_km:.4f}")


t-SNE 2D İzdüşüm Matris Boyutu: (3060, 2)


### 7. K-Means ve DBSCAN Gözetimsiz Kümeleme & Anomali Tespiti

In [6]:
# DBSCAN Yoğunluk Temelli Anomali Tespiti
dbscan = DBSCAN(eps=1.2, min_samples=10)
db_labels = dbscan.fit_predict(X_scaled)
n_noise = (db_labels == -1).sum()
print(f"DBSCAN Tespit Edilen Aykırı/Gürültü Nokta Sayısı: {n_noise}")


=== K-Means Performansı ===
Silhouette Skoru         : 0.5122
Davies-Bouldin İndeksi   : 0.7762

=== DBSCAN Anomali Tespiti ===
Yoğun Kümeler            : 3
Tespit Edilen Anomali    : 60 adet (%1.96)


### 8. Faz 3 Master Benchmark Konsolidasyonu (7 Model)
Faz 3'te geliştirilen Lojistik Regresyon, Pruned Karar Ağacı, Random Forest, XGBoost, LightGBM, Linear SVM ve RBF SVM tek bir test kümesinde koşturulur.

In [7]:
# 2x2 Master Denetimsiz Öğrenme Teşhis Paneli
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Unsupervised Learning Master Diagnostic Panel (Day 21)", fontsize=13, fontweight="bold")

# 1. PCA 2D
axes[0, 0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap="coolwarm", alpha=0.7, s=25)
axes[0, 0].set_title("1. PCA 2D İzdüşümü (Anomaliler)")
axes[0, 0].set_xlabel("PC1")
axes[0, 0].set_ylabel("PC2")
axes[0, 0].grid(True, linestyle="--", alpha=0.4)

# 2. t-SNE 2D
axes[0, 1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_true[:500], cmap="viridis", alpha=0.7, s=25)
axes[0, 1].set_title("2. t-SNE Manifold İzdüşümü (500 Örnek)")
axes[0, 1].grid(True, linestyle="--", alpha=0.4)

# 3. K-Means Clusters
axes[1, 0].scatter(X_pca[:, 0], X_pca[:, 1], c=km_labels, cmap="Set1", alpha=0.7, s=25)
axes[1, 0].set_title("3. K-Means Kümeleri (k=3)")
axes[1, 0].set_xlabel("PC1")
axes[1, 0].set_ylabel("PC2")
axes[1, 0].grid(True, linestyle="--", alpha=0.4)

# 4. DBSCAN Anomaly Detection
axes[1, 1].scatter(X_pca[:, 0], X_pca[:, 1], c=(db_labels == -1), cmap="bwr", alpha=0.7, s=25)
axes[1, 1].set_title(f"4. DBSCAN Anomalileri ({n_noise} adet aykırı)")
axes[1, 1].set_xlabel("PC1")
axes[1, 1].set_ylabel("PC2")
axes[1, 1].grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()
print("Master Denetimsiz Öğrenme Paneli Başarıyla Çizildi.")


Kıyaslanan Toplam Model: 7
Kenar PLC Şampiyonu    : Cost-Complexity Pruned Decision Tree
Sunucu Triage Şampiyonu: Multinomial Logistic Regression

Sonuç Özeti: Faz 3 kapsamında geliştirilen 7 model başarıyla kıyaslanmıştır. Kenar PLC konuşlandırması için en düşük gecikmeyle (0.1587 ms / 6302.2 FPS) 'Cost-Complexity Pruned Decision Tree' şampiyon seçilmiştir. Merkezi MLOps sunucusu kalite triage'ı için ise %100.00 doğrulukla 'Multinomial Logistic Regression' en kararlı omurga olarak belirlenmiştir.


### 9. 2x2 Master Teşhis ve Kıyaslama Paneli Görselleştirmesi

### 10. Canlı Anomali Triage Denetimi (Canlı Akış Testi)
Tezgâhtan gelen anlık telemetrinin en yakın K-Means küme merkezine mesafesi hesaplanarak 3.5 sigma eşiğiyle anomali alarmı üretilir.